In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Composition
import pandas as pd

import os
from dotenv import load_dotenv

load_dotenv()  # читает локальный файл .env, которого нет на GitHub
API_KEY = os.getenv("MP_API_KEY")

# Список запрещенных токсичных и радиоактивных элементов (Rule-based фильтр)
TOXIC_ELEMENTS = {
    "Pb", "Cd", "Hg", "As", "Tl", "Be", "Se", "Sb", "Te", "Os", "Ba",
    # Радиоактивные элементы (Tc, Pm и все элементы с Z >= 84: Po, At, Rn, Fr, Ra, Ac, Th, Pa, U, Np, Pu...)
    "Tc", "Pm", "Po", "At", "Rn", "Fr", "Ra", "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", "Cm"
}

def check_is_nontoxic(comp: Composition, oxidation_states: list = None) -> bool:
    """Проверка состава по правилам нетоксичности (без ML)."""
    elements = {el.symbol for el in comp.elements}
    # 1. Отсев по запрещенным элементам и тяжелым радиоактивным ядрам (Z >= 84)
    if elements & TOXIC_ELEMENTS:
        return False
    if any(el.Z >= 84 for el in comp.elements):
        return False
    # 2. Отсев по шестивалентному хрому Cr(VI), если известны степени окисления
    if "Cr" in elements and oxidation_states:
        if 6 in oxidation_states or 6.0 in oxidation_states:
            return False
    return True

# 1. Выгрузка оксидов из Materials Project
with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        elements=["O"],
        energy_above_hull=(0, 0.10),  # Отсев нестабильных виртуальных фаз (> 0.1 эВ/атом)
        deprecated=False,
        fields=[
            "material_id", "formula_pretty", "composition", "chemsys",
            "structure", "symmetry", "ordering", "total_magnetization",
            "energy_above_hull", "is_stable", "possible_species"
        ]
    )

records = []
for d in docs:
    comp = d.composition
    # Исключаем чистый кислород O2 и озон O3 (оставляем минимум 2 элемента)
    if len(comp.elements) < 2:
        continue
    
    # Исключаем не-оксиды, где кислород не является анионом (например, OF2)
    if "F" in {el.symbol for el in comp.elements} and len(comp.elements) == 2:
        continue

    ordering_str = str(d.ordering.value) if hasattr(d.ordering, "value") else str(d.ordering)
    if ordering_str not in {"FM", "FiM", "AFM", "NM"}:
        continue

    # Проверка наличия Cr6+ в possible_species (например, 'Cr6+')
    has_cr6 = any("Cr6+" in str(sp) for sp in (d.possible_species or []))
    ox_states = [6] if has_cr6 else []

    records.append({
        "material_id": str(d.material_id),
        "formula": d.formula_pretty,
        "reduced_formula": comp.reduced_formula,
        "chemsys": d.chemsys,  # Группа для GroupKFold (разбиение по семействам составов)
        "spacegroup_symbol": d.symmetry.symbol if d.symmetry else None,
        "spacegroup_number": d.symmetry.number if d.symmetry else None,
        "crystal_system": str(d.symmetry.crystal_system) if d.symmetry else None,
        "energy_above_hull": d.energy_above_hull,
        "is_stable": d.is_stable,
        "ordering": ordering_str,
        "target_any_magnetic": int(ordering_str in {"FM", "FiM", "AFM"}),
        "target_ferro": int(ordering_str in {"FM", "FiM"}),
        "total_magnetization": d.total_magnetization,
        "is_nontoxic": check_is_nontoxic(comp, ox_states)
    })

df_raw = pd.DataFrame(records)

# 2. Очистка дубликатов: для одного и того же состава (reduced_formula)
# сортируем по energy_above_hull и оставляем наиболее термодинамически стабильную фазу
df_clean = (
    df_raw.sort_values("energy_above_hull", ascending=True)
    .drop_duplicates(subset=["reduced_formula"], keep="first")
    .reset_index(drop=True)
)

print(f"Всего выгружено записей оксидов (E_hull <= 0.1 эВ): {len(df_raw)}")
print(f"Уникальных составов после удаления полиморфных дубликатов: {len(df_clean)}")
print("\nРаспределение классов магнитного упорядочения:")
print(df_clean["ordering"].value_counts())
print(f"\nДоля нетоксичных составов по фильтру: {df_clean['is_nontoxic'].mean():.1%}")

df_clean.to_csv("../data/oxides_dataset_v0.csv", index=False)
